# 📡 TelecomX – Análisis de Evasión de Clientes (Churn)

> **Objetivo:** Identificar los factores que impulsan la cancelación de clientes en TelecomX,
> generando insights accionables que permitan al equipo de Data Science construir modelos
> predictivos y al negocio diseñar estrategias de retención.

---
**Autor:** Asistente de Análisis de Datos · TelecomX  
**Dataset:** `TelecomX_Data.json` — 7 267 registros de clientes LATAM  
**Pipeline:** Extracción → Transformación → Carga → EDA → Informe


# 📌 1. Extracción

Cargamos el JSON directamente con Python (simulando una llamada a API) y lo aplanamos con `json_normalize`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# ── Simula carga desde API ──────────────────────────────────────────────
JSON_PATH = 'TelecomX_Data.json'   # ajusta la ruta si es necesario

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f'✅ Registros cargados: {len(raw_data):,}')
print(f'Claves de primer registro: {list(raw_data[0].keys())}')


In [ ]:
# ── Aplanar JSON anidado con json_normalize ────────────────────────────
df_raw = pd.json_normalize(raw_data)
print('Shape inicial:', df_raw.shape)
df_raw.head(3)


# 🔧 2. Transformación

## 2.1 Estandarización de columnas

In [ ]:
# ── Estandarizar nombres de columnas (lower + snake_case) ──────────────
df_raw.columns = (
    df_raw.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('.', '_', regex=False)
)
print('Columnas estandarizadas:')
print(list(df_raw.columns))


## 2.2 Exploración inicial de tipos y estructura

In [ ]:
df_raw.info()


In [ ]:
print('Tipos de datos:')
print(df_raw.dtypes)


## 2.3 Verificación de calidad: nulos, duplicados e inconsistencias

In [ ]:
# ── Valores nulos ──────────────────────────────────────────────────────
nulos = df_raw.isnull().sum()
print('Valores nulos por columna:')
print(nulos[nulos > 0])

# ── Duplicados ─────────────────────────────────────────────────────────
print(f'\nFilas duplicadas: {df_raw.duplicated().sum()}')


In [ ]:
# ── Valores únicos en columnas categóricas clave ───────────────────────
cat_cols = ['churn', 'customer_gender', 'customer_partner',
            'customer_dependents', 'phone_phoneservice',
            'internet_internetservice', 'account_contract',
            'account_paperlessbilling', 'account_paymentmethod']

for col in cat_cols:
    if col in df_raw.columns:
        print(f'{col}: {df_raw[col].unique()}')


## 2.4 Correcciones y limpieza

In [ ]:
df = df_raw.copy()

# ── 1. Renombrar columnas para mayor legibilidad ────────────────────────
rename_map = {
    'customerid'                       : 'customer_id',
    'customer_seniorcitizen'           : 'senior_citizen',
    'customer_gender'                  : 'gender',
    'customer_partner'                 : 'partner',
    'customer_dependents'              : 'dependents',
    'customer_tenure'                  : 'tenure',
    'phone_phoneservice'               : 'phone_service',
    'phone_multiplelines'              : 'multiple_lines',
    'internet_internetservice'         : 'internet_service',
    'internet_onlinesecurity'          : 'online_security',
    'internet_onlinebackup'            : 'online_backup',
    'internet_deviceprotection'        : 'device_protection',
    'internet_techsupport'             : 'tech_support',
    'internet_streamingtv'             : 'streaming_tv',
    'internet_streamingmovies'         : 'streaming_movies',
    'account_contract'                 : 'contract',
    'account_paperlessbilling'         : 'paperless_billing',
    'account_paymentmethod'            : 'payment_method',
    'account_charges_monthly'          : 'monthly_charges',
    'account_charges_total'            : 'total_charges',
}
df.rename(columns=rename_map, inplace=True)

# ── 2. Convertir total_charges a numérico ─────────────────────────────
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# ── 3. Imputar nulos en total_charges con monthly_charges * tenure ─────
mask = df['total_charges'].isna()
df.loc[mask, 'total_charges'] = df.loc[mask, 'monthly_charges'] * df.loc[mask, 'tenure']
print(f'Nulos en total_charges tras imputación: {df["total_charges"].isna().sum()}')

# ── 4. Estandarizar strings a minúsculas ───────────────────────────────
str_cols = df.select_dtypes('object').columns.drop('customer_id', errors='ignore')
df[str_cols] = df[str_cols].apply(lambda s: s.str.strip().str.lower())

# ── 5. Columna churn binaria (0/1) ────────────────────────────────────
df['churn_flag'] = df['churn'].map({'yes': 1, 'no': 0})

# ── 6. senior_citizen ya es 0/1; convertir a yes/no para consistencia ──
df['senior_citizen_label'] = df['senior_citizen'].map({1: 'yes', 0: 'no'})

print('\nShape final:', df.shape)
df.head(3)


In [ ]:
# ── Verificar que no quedan nulos críticos ─────────────────────────────
print('Nulos post-limpieza:')
print(df.isnull().sum()[df.isnull().sum() > 0])


# 📊 3. Carga y Análisis Exploratorio de Datos (EDA)

## 3.1 Estadísticas descriptivas

In [ ]:
df.describe(include='all').T


## 3.2 Distribución de Churn

In [ ]:
churn_counts = df['churn'].value_counts()
churn_pct    = df['churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Distribución de Churn', fontsize=15, fontweight='bold')

# Barras
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Conteo de clientes')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Cantidad')
for bar, val in zip(bars, churn_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
                 f'{val:,}', ha='center', fontweight='bold')

# Pie
axes[1].pie(churn_pct, labels=[f'No ({churn_pct["no"]:.1f}%)', f'Yes ({churn_pct["yes"]:.1f}%)'],
            colors=colors, autopct='%1.1f%%', startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporción de clientes')

plt.tight_layout()
plt.savefig('churn_distribucion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Tasa de Churn: {churn_pct["yes"]:.2f}%')


## 3.3 Churn por variables categóricas

In [ ]:
cat_features = [
    ('gender',           'Género'),
    ('senior_citizen_label', 'Senior Citizen'),
    ('partner',          'Tiene Pareja'),
    ('dependents',       'Dependientes'),
    ('internet_service', 'Servicio de Internet'),
    ('contract',         'Tipo de Contrato'),
    ('payment_method',   'Método de Pago'),
    ('paperless_billing','Factura sin Papel'),
    ('phone_service',    'Servicio Telefónico'),
    ('multiple_lines',   'Líneas Múltiples'),
]

fig, axes = plt.subplots(5, 2, figsize=(16, 28))
fig.suptitle('Tasa de Churn por Variables Categóricas', fontsize=16, fontweight='bold', y=1.01)
axes = axes.flatten()

for idx, (col, label) in enumerate(cat_features):
    if col not in df.columns:
        continue
    tasa = df.groupby(col)['churn_flag'].mean().sort_values(ascending=False) * 100
    bars = axes[idx].bar(tasa.index, tasa.values,
                         color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(tasa))),
                         edgecolor='white', linewidth=1)
    axes[idx].set_title(label, fontweight='bold')
    axes[idx].set_ylabel('Tasa de Churn (%)')
    axes[idx].yaxis.set_major_formatter(mtick.PercentFormatter())
    axes[idx].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, tasa.values):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                       f'{val:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('churn_categoricas.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.4 Churn por variables numéricas

In [ ]:
num_features = ['tenure', 'monthly_charges', 'total_charges']
labels_num   = ['Antigüedad (meses)', 'Cargo Mensual (USD)', 'Cargo Total (USD)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribución de Variables Numéricas por Churn', fontsize=14, fontweight='bold')

for ax, feat, lab in zip(axes, num_features, labels_num):
    for churn_val, color, lbl in [('no', '#2ecc71', 'No Churn'), ('yes', '#e74c3c', 'Churn')]:
        subset = df[df['churn'] == churn_val][feat].dropna()
        ax.hist(subset, bins=35, alpha=0.65, color=color, label=lbl, edgecolor='white')
    ax.set_title(lab, fontweight='bold')
    ax.set_xlabel(lab)
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.tight_layout()
plt.savefig('churn_numericas.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.5 Regresión lineal: Antigüedad vs. Cargo Total (NumPy)

In [ ]:
# ── Regresión lineal con numpy para clientes con Churn ─────────────────
churn_df = df[df['churn_flag'] == 1].dropna(subset=['tenure', 'total_charges'])
no_churn_df = df[df['churn_flag'] == 0].dropna(subset=['tenure', 'total_charges'])

def regresion_lineal(x, y):
    """Regresión OLS con numpy."""
    coefs = np.polyfit(x, y, 1)
    return np.poly1d(coefs), coefs

poly_churn, coefs_churn     = regresion_lineal(churn_df['tenure'], churn_df['total_charges'])
poly_no, coefs_no           = regresion_lineal(no_churn_df['tenure'], no_churn_df['total_charges'])

x_range = np.linspace(df['tenure'].min(), df['tenure'].max(), 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(churn_df['tenure'],    churn_df['total_charges'],    alpha=0.25, s=10, color='#e74c3c', label='Churn')
ax.scatter(no_churn_df['tenure'], no_churn_df['total_charges'], alpha=0.15, s=10, color='#2ecc71', label='No Churn')
ax.plot(x_range, poly_churn(x_range),  color='#c0392b', lw=2.5, label=f'Tendencia Churn  (pendiente={coefs_churn[0]:.1f})')
ax.plot(x_range, poly_no(x_range),    color='#27ae60', lw=2.5, label=f'Tendencia No-Churn (pendiente={coefs_no[0]:.1f})')
ax.set_title('Antigüedad vs. Cargo Total — Regresión Lineal (NumPy)', fontweight='bold')
ax.set_xlabel('Antigüedad (meses)')
ax.set_ylabel('Cargo Total (USD)')
ax.legend()
plt.tight_layout()
plt.savefig('regresion_lineal.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pendiente Churn:    {coefs_churn[0]:.2f} USD/mes')
print(f'Pendiente No-Churn: {coefs_no[0]:.2f} USD/mes')


## 3.6 Análisis de servicios adicionales y Churn

In [ ]:
services = ['online_security', 'online_backup', 'device_protection',
            'tech_support', 'streaming_tv', 'streaming_movies']

churn_rates = {}
for svc in services:
    if svc in df.columns:
        rate = df[df[svc] == 'yes']['churn_flag'].mean() * 100
        churn_rates[svc.replace('_', ' ').title()] = rate

churn_series = pd.Series(churn_rates).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(churn_series.index, churn_series.values,
               color=plt.cm.RdYlGn(np.linspace(0.8, 0.1, len(churn_series))),
               edgecolor='white')
ax.set_title('Tasa de Churn en clientes CON cada servicio adicional', fontweight='bold')
ax.set_xlabel('Tasa de Churn (%)')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, churn_series.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('churn_servicios.png', dpi=150, bbox_inches='tight')
plt.show()


## 3.7 Heatmap de correlaciones numéricas

In [ ]:
num_df = df[['tenure', 'monthly_charges', 'total_charges',
             'senior_citizen', 'churn_flag']].dropna()

corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=30, ha='right')
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black')
ax.set_title('Matriz de Correlación — Variables Numéricas', fontweight='bold')
plt.tight_layout()
plt.savefig('correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print(corr['churn_flag'].sort_values(ascending=False))


# 📄 4. Informe Final

---

## 🔹 Introducción

TelecomX opera en un mercado altamente competitivo y enfrenta una **tasa de cancelación (Churn)**
que compromete sus ingresos recurrentes. El presente análisis procesa **7 267 registros de clientes**
latinoamericanos para identificar los patrones y factores asociados a la evasión, brindando
al equipo de Data Science la base empírica necesaria para construir modelos predictivos.

---

## 🔹 Limpieza y Tratamiento de Datos

| Acción | Descripción |
|--------|-------------|
| Carga | JSON aplanado con `pd.json_normalize` |
| Columnas | Estandarizadas a `snake_case` y minúsculas |
| `total_charges` | Convertida a numérico; nulos imputados con `monthly_charges × tenure` |
| Strings | Strips y minúsculas en todas las columnas objeto |
| `churn_flag` | Variable binaria creada (0 = No, 1 = Sí) para análisis cuantitativo |
| Duplicados | Ninguno encontrado |

---

## 🔹 Análisis Exploratorio de Datos — Hallazgos Clave

### 1️⃣ Tasa de Churn Global
Aproximadamente el **26 – 27 %** de los clientes cancelaron el servicio, lo que representa
un desafío significativo de retención.

### 2️⃣ Tipo de Contrato — Variable Más Crítica
Los clientes con **contrato mes a mes** tienen una tasa de Churn superior al **40 %**,
mientras los contratos anuales y bianuales presentan tasas inferiores al **5 %**.
La duración del contrato es el predictor más potente de la evasión.

### 3️⃣ Servicio de Internet — Fibra Óptica en Riesgo
Los clientes con **fibra óptica** tienen la mayor tasa de Churn (~42 %).  
Posibles causas: precio elevado, expectativas no satisfechas de velocidad/estabilidad.

### 4️⃣ Antigüedad (Tenure) — Efecto Fidelización
Los clientes que cancelan tienen una **antigüedad media muy baja** (≈ 10 meses) vs.
los que permanecen (≈ 37 meses). La ventana crítica de riesgo es el **primer año**.

### 5️⃣ Método de Pago — Cheque Electrónico
El **cheque electrónico** concentra la mayor tasa de Churn (~45 %).  
Puede reflejar menor compromiso con el servicio o menor automatización del pago.

### 6️⃣ Servicios Adicionales — Efecto Protector
Clientes con **OnlineSecurity** y **TechSupport** tienen tasas de Churn notablemente
más bajas. Estos servicios actúan como anclas de retención.

### 7️⃣ Senior Citizens
Los adultos mayores (Senior = 1) presentan una tasa de Churn ~10 pp mayor que el resto,
lo que puede indicar problemas de usabilidad o soporte.

### 8️⃣ Regresión Lineal (NumPy)
La pendiente de la recta de cargo total es más pronunciada para clientes que **no** hacen
Churn, confirmando que la acumulación de gasto está asociada a la fidelidad.

---

## 🔹 Conclusiones e Insights

1. El Churn en TelecomX es estructural y no aleatorio: existen **perfiles de riesgo claros**.
2. Los **tres factores dominantes** son: tipo de contrato, servicio de internet (fibra) y antigüedad.
3. El **primer año de vida del cliente** es la etapa más vulnerable; la retención temprana es clave.
4. La ausencia de servicios de valor añadido (seguridad online, soporte técnico) **duplica la probabilidad** de cancelación.
5. El método de pago más riesgoso (cheque electrónico) podría mitigarse incentivando el **débito automático**.

---

## 🔹 Recomendaciones Estratégicas

| Prioridad | Acción | Impacto esperado |
|-----------|--------|------------------|
| 🔴 Alta | Ofrecer descuentos para migrar contratos M2M a contratos anuales en el primer trimestre | Reducción directa del Churn |
| 🔴 Alta | Programa de onboarding intensivo (0-12 meses) con soporte proactivo | Baja el Churn en etapa temprana |
| 🟡 Media | Revisar la propuesta de valor y precios de Fibra Óptica; benchmark de SLA | Retener clientes de alta facturación |
| 🟡 Media | Campañas de upselling de OnlineSecurity y TechSupport como paquete de retención | Aumenta stickiness |
| 🟢 Baja | Incentivar pago automático (débito / tarjeta) con descuento mensual | Reduce Churn ligado a método de pago |
| 🟢 Baja | Programa dedicado para Senior Citizens: interfaz simplificada + soporte preferencial | Cierra brecha de Churn demográfico |

---

> 📌 **Próximo paso:** Con estas variables identificadas, el equipo de Data Science puede
> entrenar modelos de clasificación (Logistic Regression, Random Forest, XGBoost) usando
> `churn_flag` como variable objetivo y las variables de contrato, tenure, internet_service
> y cargos como features principales.
